In [1]:
from pathlib import Path

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SENSOR_DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset.npz"
)

VIDEO_FEATURE_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "video_features_temporal"
)

In [4]:
sensor_data = np.load(SENSOR_DATASET_PATH)

y = sensor_data["y"]
groups = sensor_data["groups"]

video_feature_files = sorted(
    VIDEO_FEATURE_PATH.glob("*.npy")
)

print(len(video_feature_files))

40


In [5]:
unique_groups = np.unique(groups)

trip_labels = np.array([
    y[groups == trip][0]
    for trip in unique_groups
])

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
    stratify=trip_labels,
)

In [6]:
video_samples = []
labels = []
aligned_groups = []

for trip_id in unique_groups:

    video_trip = np.load(
        video_feature_files[trip_id]
    )

    label_trip = y[groups == trip_id]

    common = min(
        len(video_trip),
        len(label_trip),
    )

    video_samples.append(
        video_trip[:common]
    )

    labels.append(
        label_trip[:common]
    )

    aligned_groups.append(
        np.full(common, trip_id)
    )

In [7]:
video_samples = np.concatenate(video_samples)
labels = np.concatenate(labels)
aligned_groups = np.concatenate(aligned_groups)

print(video_samples.shape)
print(labels.shape)

(30560, 12, 2048)
(30560,)


In [8]:
train_mask = np.isin(
    aligned_groups,
    train_groups,
)

test_mask = np.isin(
    aligned_groups,
    test_groups,
)

X_train = video_samples[train_mask]
X_test = video_samples[test_mask]

y_train = labels[train_mask]
y_test = labels[test_mask]

print(X_train.shape)
print(X_test.shape)

(24609, 12, 2048)
(5951, 12, 2048)


In [9]:
import torch
from torch.utils.data import Dataset


class VideoDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [10]:
train_dataset = VideoDataset(
    X_train,
    y_train,
)

test_dataset = VideoDataset(
    X_test,
    y_test,
)

print(len(train_dataset))
print(len(test_dataset))

24609
5951


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

print(len(train_loader))
print(len(test_loader))

770
186


In [12]:
video, label = next(iter(train_loader))

print(video.shape)
print(label.shape)

torch.Size([32, 12, 2048])
torch.Size([32])


In [34]:
import sys

SRC = "/content/drive/MyDrive/UAH_Project/src"

if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [38]:
import importlib.util

module_path = "/content/drive/MyDrive/UAH_Project/src/video_lstm_model.py"

spec = importlib.util.spec_from_file_location(
    "video_lstm_model",
    module_path,
)

video_lstm_model = importlib.util.module_from_spec(spec)

spec.loader.exec_module(video_lstm_model)

VideoLSTMClassifier = video_lstm_model.VideoLSTMClassifier

print("VideoLSTM imported successfully!")

VideoLSTM imported successfully!


In [37]:
import os

print(os.listdir("/content/drive/MyDrive/UAH_Project/src"))

['__init__.py', 'utils.py', '__pycache__', '_sync_test.txt', 'config.py', 'preprocessor.py', 'data_loader.py', 'trainer.py', 'lstm_model.py', 'fusion_model.py', 'fusion_trainer.py', 'video_model.py', 'video_trainer.py', 'video_lstm_model.py']


In [40]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = VideoLSTMClassifier().to(device)

print(model)

VideoLSTMClassifier(
  (lstm): LSTM(2048, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [41]:
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

weights = torch.FloatTensor(
    class_weights
).to(device)

print(class_weights)

[0.81800957 0.9951474  1.29425686]


In [42]:
criterion = torch.nn.CrossEntropyLoss(
    weight=weights
)

In [43]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
)

In [44]:
EPOCHS = 20
PATIENCE = 3

In [45]:
import trainer

from trainer import fit

In [46]:
import importlib.util

module_path = "/content/drive/MyDrive/UAH_Project/src/trainer.py"

spec = importlib.util.spec_from_file_location(
    "trainer",
    module_path,
)

trainer = importlib.util.module_from_spec(spec)

spec.loader.exec_module(trainer)

fit = trainer.fit

print("Trainer imported successfully!")

Trainer imported successfully!


In [47]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
)

Epoch 1/20 | Train Loss: 0.8435 | Train Acc: 0.5794 | Val Loss: 1.4733 | Val Acc: 0.2978
Epoch 2/20 | Train Loss: 0.3881 | Train Acc: 0.8533 | Val Loss: 2.2081 | Val Acc: 0.2855
Epoch 3/20 | Train Loss: 0.1786 | Train Acc: 0.9392 | Val Loss: 2.9049 | Val Acc: 0.3351
Epoch 4/20 | Train Loss: 0.0912 | Train Acc: 0.9693 | Val Loss: 3.7682 | Val Acc: 0.3203
Epoch 5/20 | Train Loss: 0.0513 | Train Acc: 0.9841 | Val Loss: 4.7457 | Val Acc: 0.3262
Epoch 6/20 | Train Loss: 0.0292 | Train Acc: 0.9909 | Val Loss: 5.3057 | Val Acc: 0.3480
Epoch 7/20 | Train Loss: 0.0136 | Train Acc: 0.9959 | Val Loss: 6.3793 | Val Acc: 0.3346
Epoch 8/20 | Train Loss: 0.0114 | Train Acc: 0.9965 | Val Loss: 6.7745 | Val Acc: 0.3415
Epoch 9/20 | Train Loss: 0.0097 | Train Acc: 0.9973 | Val Loss: 7.3542 | Val Acc: 0.3567
Epoch 10/20 | Train Loss: 0.0083 | Train Acc: 0.9978 | Val Loss: 7.7302 | Val Acc: 0.3438
Epoch 11/20 | Train Loss: 0.0090 | Train Acc: 0.9978 | Val Loss: 7.6262 | Val Acc: 0.3761
Epoch 12/20 | Train